# SENTINEL: Fine-Tuning Small Language Models for Real-Time Fraud Detection
### Complete Pipeline: Parameter-Efficient LoRA (BF16) vs. 4-Bit QLoRA (NF4) vs. XGBoost Baseline

This self-contained notebook reproduces the end-to-end **Project SENTINEL** production pipeline:
1. **Environment Setup & Verification**: Hardware acceleration diagnostics and pinned dependency installation.
2. **Pre-Flight Health Verification**: Verifies GPU CUDA availability, HuggingFace weights cache, and dataset integrity.
3. **Parameter-Efficient LoRA Fine-Tuning (BF16)**: Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` with rank-16 adapters.
4. **Quantized QLoRA Fine-Tuning (4-Bit NF4)**: Fine-tunes quantized base weights using BitsAndBytes NF4 and gradient checkpointing.
5. **Out-of-Time Holdout Evaluation**: Evaluates precision, recall, false positive rates, ECE, and PR-AUC on 13,889 holdout transactions.
6. **Dynamic Acceptance Matrix**: Derives production deployment verdict based on strict SLA gates.
7. **Actionable Next-Training Experiment**: Runs asymmetric loss weighting (`pos_weight=2.5`) to capture subtle fraud.

## 1. Repository Setup & Environment Installation

In [ ]:
# Clone repository if running in a fresh Google Colab environment
import os
if not os.path.exists("sentinel"):
    !git clone https://github.com/pavanmajji18/sentinel-finetuning-lora-qlora.git
    %cd sentinel-finetuning-lora-qlora

# Verify GPU hardware
!nvidia-smi

In [ ]:
# Install pinned dependencies matching the production environment
!pip install -q -r requirements.txt

## 2. Infrastructure Configuration & Pre-Flight Checks

In [ ]:
# Configure optional Supabase data endpoints (falls back to local data if present)
os.environ["SUPABASE_URL"] = os.getenv("SUPABASE_URL", "https://your-project.supabase.co")
os.environ["SUPABASE_KEY"] = os.getenv("SUPABASE_KEY", "your-supabase-key")
os.environ["SENTINEL_API_KEY"] = os.getenv("SENTINEL_API_KEY", "sentinel_sk_live_demo")
print("✓ Storage endpoints and authentication headers configured.")

In [ ]:
# Run the automated pre-flight health check script
!bash hyperstack/check.sh

## 3. Parameter-Efficient LoRA Fine-Tuning (BF16)

In [ ]:
# Execute Parameter-Efficient LoRA fine-tuning on Qwen2.5-0.5B
# Expected runtime: ~7.8 min on NVIDIA L40 (Peak VRAM: 2.38 GiB w/ gradient checkpointing)
!python -m sentinel.train --tag colab --method lora --epochs 2.0 --batch-size 16 --lr 2e-4

## 4. Memory-Optimized 4-Bit QLoRA Fine-Tuning (NF4)

In [ ]:
# Execute 4-bit Quantized LoRA fine-tuning on Qwen2.5-0.5B
# Expected runtime: ~8.5 min on NVIDIA L40 (Peak VRAM: 1.62 GiB w/ gradient checkpointing)
!python -m sentinel.train --tag colab --method qlora --epochs 2.0 --batch-size 16 --lr 2e-4

## 5. Out-of-Time Holdout Evaluation & Multi-Candidate Benchmark

In [ ]:
# Run the benchmark suite: telemetry comparison, holdout evaluation, dynamic acceptance matrix, and case studies
!python run_evaluation.py

## 6. Live Inference Pipeline & Platt Probability Calibration Demonstration

In [ ]:
# Demonstrate live raw logit scoring, Platt calibration, threshold routing, and reason code generation
!python run_evaluation.py --demo-live

## 7. Actionable Next-Training Experiment: Asymmetric Loss Weighting (`pos_weight=2.5`)

In [ ]:
# Inspect the actionable next-experiment protocol, hyperparameter grid, and HyperStack compute economics
!python run_evaluation.py --next-experiment